In [1]:
import pyspark
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
filepath = 'D:/Dataset/sf-fire-calls.csv'


In [2]:
def create_sparkSession():
    spark = SparkSession.builder.appName('Fire Example').getOrCreate()
    return spark


In [3]:
def clean_dataset(df):
    df1 = df.withColumn('Date', to_date(col('CallDate'), 'MM/dd/yyyy')).drop('CallDate')
    df2 = df1.withColumn('Year',year(col('Date')))\
    .withColumn('Month',month(col('Date')))\
    .withColumn('Week',weekofyear(col('Date')))  
    return df2


In [5]:
spark = create_sparkSession()
df = create_dataframe(spark, filepath)
df = clean_dataset(df)
df.printSchema()


root
 |-- CallType: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Zipcode: integer (nullable = true)
 |-- Neighborhood: string (nullable = true)
 |-- Delay: double (nullable = true)
 |-- Date: date (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Month: integer (nullable = true)
 |-- Week: integer (nullable = true)



In [4]:
def create_dataframe(spark, filepath):
    df = spark.read.csv(filepath, header=True, inferSchema=True)
    df1= df.select('CallType','CallDate','City','Zipcode','Neighborhood', 'Delay')
    return df1


In [ ]:
df.show()

In [6]:
# create a user defined function
def mapSeason(data):
    if 2 < data < 6:
        return 'Spring'
    elif 5 < data < 9:
        return 'Summer'
    elif 8 < data < 12:
        return 'Autumn'
    else:
        return 'Winter'
seasonUDF = udf(mapSeason, StringType())
clean_df = df.withColumn('Season', seasonUDF(col('Month')))
clean_df.show()


+----------------+----+-------+--------------------+---------+----------+----+-----+----+------+
|        CallType|City|Zipcode|        Neighborhood|    Delay|      Date|Year|Month|Week|Season|
+----------------+----+-------+--------------------+---------+----------+----+-----+----+------+
|  Structure Fire|  SF|  94109|     Pacific Heights|     2.95|2002-01-11|2002|    1|   2|Winter|
|Medical Incident|  SF|  94124|Bayview Hunters P...|      4.7|2002-01-11|2002|    1|   2|Winter|
|Medical Incident|  SF|  94102|          Tenderloin|2.4333334|2002-01-11|2002|    1|   2|Winter|
|    Vehicle Fire|  SF|  94110|      Bernal Heights|      1.5|2002-01-11|2002|    1|   2|Winter|
|          Alarms|  SF|  94109|    Western Addition|3.4833333|2002-01-11|2002|    1|   2|Winter|
|  Structure Fire|  SF|  94105|Financial Distric...|     1.75|2002-01-11|2002|    1|   2|Winter|
|          Alarms|  SF|  94112|Oceanview/Merced/...|2.7166667|2002-01-11|2002|    1|   2|Winter|
|          Alarms|  SF|  94102

In [ ]:
#1.	Get yearly count of fire calls
clean_df.select('Year').groupBy('Year').count().orderBy('Year', ascending=True).show()


In [ ]:
#2.	What were all the different types of fire calls in 2018?
clean_df.select('CallType').where(col('Year') == 2018).distinct().show(truncate=False)


In [ ]:

#3.	Which week in the year in 2018 had the most fire calls?
clean_df.select('Week')\
    .where(col('Year') == 2018)\
    .groupBy('Week')\
    .count()\
    .orderBy('count', ascending=False).collect()[0][0]


In [ ]:
max_month = clean_df.select('Week')\
    .where(col('Year') == 2018)\
    .groupBy('Week')\
    .count()
max_month.select('Week','count').filter(col('count') == max_month.agg({'count':'max'}).collect()[0][0]).collect()[0][0]


In [ ]:
# Running SQL query in pyspark
clean_df.createOrReplaceTempView('Fire')
spark.sql('select Year from Fire where Delay > 520').show()


In [7]:
from pyspark.sql.window import Window
#6.	Give top five fire call types for every season of selected year 
#(seasons are like Spring, summer, fall winter etc).

season_count = clean_df.select('CallType', 'Season')\
    .filter(col('Year') == 2014)\
    .groupBy('Season', 'CallType')\
    .count()

winSeason = Window.partitionBy('Season').orderBy(col('count').desc())
top_five = season_count.withColumn('Rank', dense_rank().over(Window.partitionBy('Season').orderBy(col('count').desc())))\
    .orderBy('Season', 'Rank',ascending=[1,1])\
    .filter(col('Rank') < 6)
top_five.show()

+------+--------------------+-----+----+
|Season|            CallType|count|Rank|
+------+--------------------+-----+----+
|Autumn|    Medical Incident| 1832|   1|
|Autumn|      Structure Fire|  319|   2|
|Autumn|              Alarms|  314|   3|
|Autumn|   Traffic Collision|  120|   4|
|Autumn|Citizen Assist / ...|   26|   5|
|Spring|    Medical Incident| 1801|   1|
|Spring|      Structure Fire|  314|   2|
|Spring|              Alarms|  281|   3|
|Spring|   Traffic Collision|  112|   4|
|Spring|Citizen Assist / ...|   36|   5|
|Summer|    Medical Incident| 1770|   1|
|Summer|      Structure Fire|  322|   2|
|Summer|              Alarms|  269|   3|
|Summer|   Traffic Collision|  112|   4|
|Summer|        Outside Fire|   37|   5|
|Winter|    Medical Incident| 1773|   1|
|Winter|              Alarms|  337|   2|
|Winter|      Structure Fire|  310|   3|
|Winter|   Traffic Collision|  111|   4|
|Winter|Citizen Assist / ...|   36|   5|
+------+--------------------+-----+----+

